# 変数の作成

`fetch_ch01-24.ipynb` が作った年ごとファイル（`data/by_year_allp/trade_{年}.csv.gz`）から、
分析用の変数を組み立てる。

元データは **HS 1〜24類（食料・農水産物）× 2012〜2025年 × 全報告国 × 全相手国 × 輸出**、
金額は `primaryValue`（USD、輸出なので FOB 建て）。

## 変数一覧

| 変数名 | 定義 | 単位 | 粒度 |
|---|---|---|---|
| `japancountry` | **日本から輸入国 *m* への輸出額の合計**。日本を報告国、*m* を相手国とする行を、HS 1〜24類の全品目について合算したもの | USD | 年 × 相手国 *m* |
| `worldcountry` | **世界各国から輸入国 *m* への総輸出額**。全報告国を対象に、*m* を相手国とする行を HS 1〜24類の全品目について合算したもの。*m* の輸入市場規模にあたり、`japancountry` はこの一部 | USD | 年 × 相手国 *m* |
| `japanx` | **日本が *m* へ輸出している品目に限定した、世界各国から *m* への輸出額**。その年に日本が *m* へ売っている HS6桁の集合を特定し、その品目について全報告国 → *m* を合算したもの。日本が競合している到達可能な市場規模。定義上 `japancountry ≤ japanx ≤ worldcountry` | USD | 年 × 相手国 *m* |

いずれも `partnerDesc == "World"`（全世界合計の行）は除外して集計する。
残すと個別相手国の合計と二重計上になるため。

各変数は3つの形で保存する。

| 形 | ファイル | 使いどころ |
|---|---|---|
| 年ごとに1ファイル | `{変数名}/{変数名}_{年}.csv` | その年だけ扱いたいとき |
| 横持ち（相手国 × 年） | `{変数名}_wide.csv` | 国ごとの推移を横に並べて見たいとき |
| 縦持ち（年 × 相手国） | `{変数名}.csv` | 回帰などパネル分析にそのまま使う |

変数を増やすときは末尾の雛形を複製する。

## 1. セットアップ

In [11]:
from pathlib import Path

import pandas as pd

SRC   = Path("data/by_year_allp")     # 年ごとファイルの置き場所
OUT   = Path("data/variables")        # 変数の保存先
YEARS = [str(y) for y in range(2012, 2026)]

OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

files = {y: SRC / f"trade_{y}.csv.gz" for y in YEARS}
lack  = [y for y, p in files.items() if not p.exists()]
if lack:
    raise FileNotFoundError(
        f"年ごとファイルが無い: {lack}\n"
        "→ fetch_ch01-24.ipynb のセクション7を実行して作成すること")
print(f"入力: {SRC.resolve()}  ({len(files)} 年分)")

入力: /Users/nakasukadaiki/Projects/comtrade/data/by_year_allp  (14 年分)


## 2. 元データの読み込み

全体は約1,290万行あるため、**必要な報告国だけを読み込みながら絞る**。
日本の行だけなら14年で7万行程度に収まる。

In [12]:
JAPAN = 392        # 日本の reporterCode

# dtype 指定は必須。省くと 0201 が 201 になり先頭ゼロが落ちる。
DTYPE = {"cmdCode": str, "classificationCode": str}
USE   = ["refYear", "reporterCode", "reporterDesc", "flowDesc",
         "partnerCode", "partnerDesc", "cmdCode", "primaryValue"]


def load_reporter(reporter_code):
    """指定した報告国の行だけを全年ぶん読み込む。"""
    out = []
    for y in YEARS:
        d = pd.read_csv(files[y], dtype=DTYPE, usecols=USE)
        out.append(d[d["reporterCode"] == reporter_code])
    return pd.concat(out, ignore_index=True)


jp = load_reporter(JAPAN)
print(f"日本の行数: {len(jp):,}")
print(f"年        : {sorted(jp['refYear'].unique())}")
print(f"フロー    : {sorted(jp['flowDesc'].unique())}")
print(f"相手国数  : {jp['partnerCode'].nunique()}（World 含む）")

日本の行数: 96,960
年        : [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
フロー    : ['Export']
相手国数  : 195（World 含む）


## 3. 変数① `japancountry`

**日本から輸入国 *m* への輸出額の合計**（HS 1〜24類を合算）。

- 単位は USD（名目）。`primaryValue` は輸出なので FOB 建て。
- `partnerDesc == "World"` は**全世界合計の行**なので除外する。
  残さないと「個別国の合計」と「World」が二重に入る。
- 年ごとに1行 × 相手国。パネルデータとして使える形にする。

In [ ]:
VALUE = "primaryValue"     # 輸出額（USD）

# World（全世界合計）を除いた、実在する相手国のみ
jp_m = jp[jp["partnerDesc"] != "World"]

japancountry = (jp_m
                .groupby(["refYear", "partnerCode", "partnerDesc"], as_index=False)[VALUE]
                .sum()
                .rename(columns={"refYear": "year", VALUE: "japancountry"})
                .sort_values(["year", "japancountry"], ascending=[True, False])
                .reset_index(drop=True))

print(f"{len(japancountry):,} 行（年 × 相手国）")
japancountry.head(10)

### 検算

除外した `World` 行の合計と、相手国別の合計が一致するはず。
ズレる場合は、報告国が World 行だけを出して内訳を出していない年がある等の理由が考えられる。

In [ ]:
world = (jp[jp["partnerDesc"] == "World"]
         .groupby("refYear", as_index=False)[VALUE].sum()
         .rename(columns={"refYear": "year", VALUE: "world_total"}))
detail = japancountry.groupby("year", as_index=False)["japancountry"].sum()

chk = world.merge(detail, on="year")
chk["差"]     = chk["world_total"] - chk["japancountry"]
chk["差の率"] = (chk["差"] / chk["world_total"] * 100).round(2)
chk

## 4. 確認

In [ ]:
# 上位10か国（直近年）
last = japancountry["year"].max()
top = japancountry[japancountry["year"] == last].nlargest(10, "japancountry").copy()
top["百万USD"] = (top["japancountry"] / 1e6).round(1)
print(f"{last}年 日本の農水産物・食品（HS1〜24類）輸出先 上位10")
top[["partnerDesc", "japancountry", "百万USD"]].reset_index(drop=True)

In [ ]:
# 主要国の推移（横持ちに変換）
wide = japancountry.pivot(index="year", columns="partnerDesc", values="japancountry")
cols = (japancountry.groupby("partnerDesc")["japancountry"].sum()
        .nlargest(8).index.tolist())
(wide[cols] / 1e6).round(1)      # 単位: 百万USD

## 5. 保存 — 年ごとに分けて出力

同じ変数を用途に応じて3つの形で出す。

| 形 | ファイル | 使いどころ |
|---|---|---|
| **年ごとに1ファイル** | `japancountry/japancountry_{年}.csv` | その年だけ扱いたいとき |
| **横持ち（相手国 × 年）** | `japancountry_wide.csv` | 国ごとの推移を横に並べて見たいとき |
| 縦持ち（年 × 相手国） | `japancountry.csv` | 回帰などパネル分析にそのまま使う |

In [ ]:
# ① 年ごとに1ファイル
YEAR_DIR = OUT / "japancountry"
YEAR_DIR.mkdir(parents=True, exist_ok=True)

for y, g in japancountry.groupby("year"):
    p = YEAR_DIR / f"japancountry_{y}.csv"
    (g.drop(columns="year")
      .sort_values("japancountry", ascending=False)
      .reset_index(drop=True)
      .to_csv(p, index=False, encoding="utf-8-sig"))
    print(f"{y}: {len(g):>3} か国 → {p.name}")

print(f"\n{len(list(YEAR_DIR.glob('*.csv')))} ファイル: {YEAR_DIR.resolve()}")

In [ ]:
# ② 横持ち（相手国 × 年）— 国ごとの推移が1行で読める
japancountry_wide = (japancountry
                     .pivot(index=["partnerCode", "partnerDesc"],
                            columns="year", values="japancountry")
                     .reset_index())
japancountry_wide.columns.name = None

p = OUT / "japancountry_wide.csv"
japancountry_wide.to_csv(p, index=False, encoding="utf-8-sig")
print("保存:", p.resolve(), f"({len(japancountry_wide)} か国 × {len(YEARS)} 年)")

# 日本→中国の例（欠損は「その年に報告がない」を意味する）
japancountry_wide[japancountry_wide["partnerDesc"] == "China"]

In [ ]:
# ③ 縦持ち（パネル形式）
p = OUT / "japancountry.csv"
japancountry.to_csv(p, index=False, encoding="utf-8-sig")
print("保存:", p.resolve(), f"({len(japancountry):,} 行)")

## 6. 変数② `worldcountry`

**世界各国から輸入国 *m* への総輸出額**（全報告国 × HS 1〜24類を合算）。年 × 相手国。

`japancountry` が「日本 → m」なのに対し、こちらは「世界全体 → m」。
つまり m の（この品目群における）**輸入市場の規模**にあたる。

### 集計の前提

- **全報告国を合算する。** 日本も含む。日本を除いた「日本以外の世界」が必要な場合は
  下のセルの `EXCLUDE_REPORTER` に `392` を入れる。
- `partnerDesc == "World"` の行は除外する（全世界合計なので m ではない）。
- 地域集計による二重計上は起きない。参照表で `isGroup=True` の報告国のうち
  データに現れるのは `Other Asia, nes`（490）だけで、これは台湾を指す個別の報告経済であり、
  他の報告国を束ねた集計ではない。`ASEAN` / `European Union` は報告国として出現しない。
- `reporterCode == partnerCode`（自国向け）の行は存在しない。

### 注意 — これは「m の輸入額」そのものではない

各国が申告した**輸出額（FOB）の積み上げ**なので、m 自身が申告する輸入額（CIF）とは一致しない。
運賃・保険料の差、報告漏れ、計上時期のズレがあるため、一般に輸入額のほうが大きくなる。

In [13]:
EXCLUDE_REPORTER = None      # 例: 392 とすると「日本を除く世界」になる

# 全報告国が対象なので、年ごとに読んで集計し、結果だけ持ち越す（全件を保持しない）
parts = []
for y in YEARS:
    d = pd.read_csv(files[y], dtype=DTYPE, usecols=USE)
    d = d[d["partnerDesc"] != "World"]                  # 全世界合計の行を除く
    if EXCLUDE_REPORTER is not None:
        d = d[d["reporterCode"] != EXCLUDE_REPORTER]
    parts.append(d.groupby(["refYear", "partnerCode", "partnerDesc"],
                           as_index=False)[VALUE].sum())
    print(f"{y}: 報告国 {d['reporterCode'].nunique():>3} / 相手国 {d['partnerCode'].nunique():>3}")

worldcountry = (pd.concat(parts, ignore_index=True)
                .groupby(["refYear", "partnerCode", "partnerDesc"], as_index=False)[VALUE].sum()
                .rename(columns={"refYear": "year", VALUE: "worldcountry"})
                .sort_values(["year", "worldcountry"], ascending=[True, False])
                .reset_index(drop=True))

print(f"\n{len(worldcountry):,} 行（年 × 相手国）")
worldcountry.head(10)

2012: 報告国 173 / 相手国 242
2013: 報告国 174 / 相手国 242
2014: 報告国 173 / 相手国 242
2015: 報告国 175 / 相手国 242
2016: 報告国 178 / 相手国 241
2017: 報告国 181 / 相手国 242
2018: 報告国 178 / 相手国 242
2019: 報告国 172 / 相手国 243
2020: 報告国 169 / 相手国 241
2021: 報告国 170 / 相手国 243
2022: 報告国 167 / 相手国 243
2023: 報告国 165 / 相手国 242
2024: 報告国 138 / 相手国 244
2025: 報告国  93 / 相手国 242

3,391 行（年 × 相手国）


,year,partnerCode,partnerDesc,worldcountry
0,2012,842,USA,1.198226e+11
1,2012,276,Germany,9.501649e+10
2,2012,156,China,8.735759e+10
3,2012,392,Japan,6.772340e+10
4,2012,528,Netherlands,6.154601e+10
5,2012,826,United Kingdom,5.897289e+10
6,2012,251,France,5.832498e+10
7,2012,380,Italy,4.756564e+10
8,2012,643,Russian Federation,3.825075e+10
9,2012,56,Belgium,3.790171e+10


### 検算

`japancountry`（日本 → m）は `worldcountry`（世界 → m）の一部なので、
**すべての (年, m) で `japancountry` ≤ `worldcountry`** が成り立つはず。

In [14]:
chk2 = japancountry.merge(worldcountry, on=["year", "partnerCode", "partnerDesc"], how="left")
bad = chk2[chk2["japancountry"] > chk2["worldcountry"] + 1e-6]
print("japancountry > worldcountry となる行:", len(bad), " ← 0 であること")

chk2["日本シェア%"] = (chk2["japancountry"] / chk2["worldcountry"] * 100).round(2)
print()
print("年ごとの世界合計と日本シェア:")
g = chk2.groupby("year").agg(世界=("worldcountry", "sum"), 日本=("japancountry", "sum"))
g["日本シェア%"] = (g["日本"] / g["世界"] * 100).round(3)
(g / [1e9, 1e9, 1]).round(3).rename(columns={"世界": "世界(10億USD)", "日本": "日本(10億USD)"})

japancountry > worldcountry となる行: 0  ← 0 であること

年ごとの世界合計と日本シェア:


,世界(10億USD),日本(10億USD),日本シェア%
year,,,
2012,1286.333,4.936,0.384
2013,1374.770,4.946,0.360
2014,1406.468,5.028,0.357
2015,1265.011,5.396,0.427
2016,1300.695,6.094,0.469
2017,1395.824,6.298,0.451
2018,1459.085,7.292,0.500
2019,1459.729,7.487,0.513
2020,1497.762,7.986,0.533


In [15]:
# 保存（japancountry と同じ3形式）
VAR = "worldcountry"
tbl = worldcountry

# ① 年ごと
d = OUT / VAR
d.mkdir(parents=True, exist_ok=True)
for y, g in tbl.groupby("year"):
    (g.drop(columns="year").sort_values(VAR, ascending=False).reset_index(drop=True)
      .to_csv(d / f"{VAR}_{y}.csv", index=False, encoding="utf-8-sig"))
print(f"① 年ごと {len(list(d.glob('*.csv')))} ファイル: {d.resolve()}")

# ② 横持ち（相手国 × 年）
wide = tbl.pivot(index=["partnerCode", "partnerDesc"], columns="year", values=VAR).reset_index()
wide.columns.name = None
wide.to_csv(OUT / f"{VAR}_wide.csv", index=False, encoding="utf-8-sig")
print(f"② 横持ち: {VAR}_wide.csv ({len(wide)} か国)")

# ③ 縦持ち
tbl.to_csv(OUT / f"{VAR}.csv", index=False, encoding="utf-8-sig")
print(f"③ 縦持ち: {VAR}.csv ({len(tbl):,} 行)")

wide[wide["partnerDesc"] == "China"]

① 年ごと 14 ファイル: /Users/nakasukadaiki/Projects/comtrade/data/variables/worldcountry
② 横持ち: worldcountry_wide.csv (246 か国)
③ 縦持ち: worldcountry.csv (3,391 行)


,partnerCode,partnerDesc,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
44,156,China,8.735759e+10,9.993495e+10,9.916396e+10,9.792714e+10,1.026290e+11,1.161787e+11,1.238056e+11,1.400735e+11,1.608295e+11,1.913366e+11,1.994634e+11,1.988498e+11,1.654340e+11,1.408052e+11


## 7. 変数③ `japanx`

**日本が輸入国 *m* へ輸出している品目に限定した、世界各国から *m* への輸出額**。

`worldcountry` は m の輸入市場を全品目で測るが、その中には日本が扱っていない品目も含まれる。
`japanx` は**日本が実際に m へ売っている品目だけ**に絞って世界の輸出額を合計するので、
日本が競合している市場の規模、つまり**到達可能な市場規模**を表す。

### 手順

1. **品目集合を特定** — その年に日本が m へ輸出している HS6桁コードの集合 *S(年, m)* を作る。
2. **世界の輸出額を合計** — 全報告国について、相手国が m かつ品目が *S(年, m)* に含まれる行を合算する。

品目集合は**年ごと・相手国ごとに別々**に作る。日本の品目構成は年で変わり、
相手国によっても違うため。全期間で固定した品目バスケットが必要な場合は、
下のセルの `FIXED_BASKET` を `True` にする。

### 大小関係

定義上、必ず次が成り立つ。

```
japancountry  ≤  japanx  ≤  worldcountry
（日本→m）      （世界→m、        （世界→m、
                 日本の品目のみ）    全品目）
```

日本の行には金額0・欠損がなく、`(相手国, 品目)` の組も一意なので、
集合の定義に曖昧さはない（2022年で6,933組）。

In [ ]:
FIXED_BASKET = False    # True にすると全期間共通の品目バスケットを使う

# 全期間共通バスケット用（FIXED_BASKET=True のときだけ使う）
if FIXED_BASKET:
    basket = (jp[jp["partnerDesc"] != "World"][["partnerCode", "cmdCode"]]
              .drop_duplicates())

parts = []
for y in YEARS:
    d = pd.read_csv(files[y], dtype=DTYPE, usecols=USE)
    d = d[d["partnerDesc"] != "World"]                 # 全世界合計の行を除く

    # ① 日本が m へ輸出している品目の集合（相手国 × 品目の組）
    if FIXED_BASKET:
        keys = basket
    else:
        keys = (d[d["reporterCode"] == JAPAN][["partnerCode", "cmdCode"]]
                .drop_duplicates())

    # ② その組に該当する全報告国の行だけを残して合算
    hit = d.merge(keys, on=["partnerCode", "cmdCode"], how="inner")
    g = (hit.groupby(["refYear", "partnerCode", "partnerDesc"], as_index=False)
            .agg(japanx=(VALUE, "sum"), n_cmd=("cmdCode", "nunique")))
    parts.append(g)
    print(f"{y}: 日本の(相手国,品目) {len(keys):>6,} 組 → 該当 {len(hit):>7,} 行 / 相手国 {len(g):>3}")

japanx = (pd.concat(parts, ignore_index=True)
          .rename(columns={"refYear": "year"})
          .sort_values(["year", "japanx"], ascending=[True, False])
          .reset_index(drop=True))

print(f"\n{len(japanx):,} 行（年 × 相手国）")
japanx.head(10)

### 検算

`japancountry ≤ japanx ≤ worldcountry` が全行で成り立つか確認する。
`japanx / worldcountry` は「日本が扱う品目が m の輸入市場のどれだけを覆っているか」、
`japancountry / japanx` は「その市場で日本が取れているシェア」と読める。

In [ ]:
chk3 = (japancountry
        .merge(japanx[["year", "partnerCode", "japanx", "n_cmd"]],
               on=["year", "partnerCode"], how="inner")
        .merge(worldcountry[["year", "partnerCode", "worldcountry"]],
               on=["year", "partnerCode"], how="left"))

eps = 1e-6
ng1 = (chk3["japancountry"] > chk3["japanx"] + eps).sum()
ng2 = (chk3["japanx"] > chk3["worldcountry"] + eps).sum()
print(f"japancountry > japanx    : {ng1}  ← 0 であること")
print(f"japanx > worldcountry    : {ng2}  ← 0 であること")
if ng1 or ng2:
    raise RuntimeError("大小関係が崩れている。集合の作り方を確認すること。")
print("大小関係 OK")

chk3["カバー率%"]  = (chk3["japanx"] / chk3["worldcountry"] * 100).round(2)
chk3["日本シェア%"] = (chk3["japancountry"] / chk3["japanx"] * 100).round(2)

print()
print("年ごとの集計（10億USD）:")
g = chk3.groupby("year").agg(japancountry=("japancountry", "sum"),
                             japanx=("japanx", "sum"),
                             worldcountry=("worldcountry", "sum"))
out = (g / 1e9).round(2)
out["カバー率%"]  = (g["japanx"] / g["worldcountry"] * 100).round(1)
out["日本シェア%"] = (g["japancountry"] / g["japanx"] * 100).round(2)
out

In [ ]:
# 主要国の内訳（直近年）
last = chk3["year"].max()
cols = ["partnerDesc", "n_cmd", "japancountry", "japanx", "worldcountry",
        "カバー率%", "日本シェア%"]
view = chk3[chk3["year"] == last].nlargest(10, "japancountry")[cols].copy()
for c in ["japancountry", "japanx", "worldcountry"]:
    view[c] = (view[c] / 1e6).round(1)     # 百万USD
print(f"{last}年 上位10か国（金額は百万USD、n_cmd は日本が輸出している品目数）")
view.reset_index(drop=True)

In [ ]:
# 保存（他の変数と同じ3形式）
VAR = "japanx"
tbl = japanx

d_ = OUT / VAR
d_.mkdir(parents=True, exist_ok=True)
for y, g in tbl.groupby("year"):
    (g.drop(columns="year").sort_values(VAR, ascending=False).reset_index(drop=True)
      .to_csv(d_ / f"{VAR}_{y}.csv", index=False, encoding="utf-8-sig"))
print(f"① 年ごと {len(list(d_.glob('*.csv')))} ファイル: {d_.resolve()}")

wide = tbl.pivot(index=["partnerCode", "partnerDesc"], columns="year", values=VAR).reset_index()
wide.columns.name = None
wide.to_csv(OUT / f"{VAR}_wide.csv", index=False, encoding="utf-8-sig")
print(f"② 横持ち: {VAR}_wide.csv ({len(wide)} か国)")

tbl.to_csv(OUT / f"{VAR}.csv", index=False, encoding="utf-8-sig")
print(f"③ 縦持ち: {VAR}.csv ({len(tbl):,} 行)")

wide[wide["partnerDesc"] == "China"]

## 8. 変数を追加するときの雛形

同じ `jp`（または `load_reporter()` で読んだ他国のデータ）から派生させる。

```python
# 例: 日本の品目別輸出額（年 × HS6桁）
japancmd = (jp[jp["partnerDesc"] == "World"]
            .groupby(["refYear", "cmdCode"], as_index=False)["primaryValue"]
            .sum()
            .rename(columns={"refYear": "year", "primaryValue": "japancmd"}))

# 例: 日本から相手国への品目別輸出額（年 × 相手国 × HS6桁）
japancountrycmd = (jp_m
                   .groupby(["refYear", "partnerCode", "cmdCode"], as_index=False)["primaryValue"]
                   .sum()
                   .rename(columns={"refYear": "year", "primaryValue": "japancountrycmd"}))
```

日本以外の報告国が必要なら `load_reporter(コード)` を使う。コードは
`data/hs_codes.csv` ではなく `comtrade_start.ipynb` の `getReference("reporter")` で調べられる。

### 注意点

- **`World` を混ぜない。** 個別相手国と合算すると必ず二重計上になる。
- **`cifvalue` は輸出データではほぼ空。** 金額は `primaryValue`（= FOB）を使う。
- **直近年は報告が出揃っていない。** 2024〜2025年は報告国数が少なく、
  時系列比較ではそのまま使うと過小評価になる。
- **HS 版が年で変わる。** 品目コード単位で年を跨いで追う場合、統廃合されたコードに注意
  （例: `010110` → `010121` / `010129`）。`classificationCode` で報告版を判別できる。